# Week 1 — Run it, then discover a real truth yourself

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/01_first_look_and_discovery.ipynb?flush_cache=true)

By the end of this notebook you will have:

1. **Run a real ML pipeline** on real (anonymized) search data and watched a learned model beat a hand-written rule.
2. **Rediscovered a real finding yourself** in ~10 lines of pandas.

No prior ML needed. Everything runs on the small anonymized dataset that ships with this repo — no credentials, no private data.


## 0. Setup (Colab or local)

On Colab this clones the repo and installs requirements. Locally it just moves to the repo root.

**Method, step by step:** the first cell finds the project's top folder (cloning it on Colab, where the notebook starts on an empty machine) and stops with a clear message if the starter CSV is missing. The second cell loads the tools and defines two small helpers, `show()` and `show_table()`, so every result is displayed as a titled table in the same layout, plus the chart theme used by the one chart below.


In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

In [ ]:
import json
import subprocess
import tempfile
import urllib.request

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from matplotlib import font_manager

pd.set_option("display.width", 160)
NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))

# Emphasis theme: cividis navy for the story, grays for what should recede.
INK, MUTED, FAINT, GRID, SURFACE = "#111111", "#5f5e5a", "#9a9893", "#eeede9", "#ffffff"
NAVY, GRAY, PALE = mpl.colormaps["cividis"](0.05), "#9a9893", "#d4d2cc"


def load_inter() -> str:
    """Register the Inter font for this session and return the font family to draw with.

    The 400 and 600 weights are cached in the system temp folder, never in the project.
    If the download is unavailable, the chart falls back to matplotlib's default font,
    so the notebook still runs offline.
    """
    cache = os.path.join(tempfile.gettempdir(), "inter-font")
    os.makedirs(cache, exist_ok=True)
    try:
        for weight in (400, 600):
            path = os.path.join(cache, f"inter-{weight}.ttf")
            if not os.path.exists(path):
                url = f"https://cdn.jsdelivr.net/fontsource/fonts/inter@latest/latin-{weight}-normal.ttf"
                urllib.request.urlretrieve(url, path)
            font_manager.fontManager.addfont(path)
    except OSError:
        return "DejaVu Sans"
    return "Inter"


plt.rcParams.update({"font.family": load_inter()})

## 1. Run the whole pipeline

This runs `scripts/run_all.py`: prepare features → baseline rule → train 3 models → evaluate → PDF.
It takes ~1 minute on the 30,000-row sample.

**Method:** the next cell runs the pipeline as a separate process, which takes about a minute, and then reads its log and shows one table row per step: what the step did and what it produced, with file paths relative to the project folder. It stops with an error if any step fails, so a finished cell means every output file was written.


In [ ]:
# Run the pipeline as a separate process and keep its log, so the five steps can be shown as one table.
run = subprocess.run(
    [sys.executable, "scripts/run_all.py"],
    check=False, capture_output=True, text=True, encoding="utf-8",
    env={**os.environ, "PYTHONIOENCODING": "utf-8"},
)
assert run.returncode == 0, "The pipeline stopped before finishing:\n" + run.stderr[-1500:]

# Turn the log into one row per step, with file paths shown relative to the project folder.
steps = []
for line in run.stdout.splitlines():
    line = line.replace(os.getcwd() + os.sep, "").strip()
    if line.startswith("▶ Step"):
        title, _, what = line[2:].partition(" — ")
        steps.append({"Step": title.replace("Step ", ""), "What it did": what, "Result": []})
    elif steps and line and not line.startswith("="):
        steps[-1]["Result"].append(line)

show_table("The five pipeline steps and what each one produced", pd.DataFrame({
    "Step": [step["Step"] for step in steps],
    "What it did": [step["What it did"] for step in steps],
    "Result": ["; ".join(step["Result"]) for step in steps],
}).set_index("Step"))

### What just happened?

The pipeline ranked every page for "refresh review" two ways: a **hand-written rule baseline** and a **learned model**. Let's compare them on **Precision@50** — of the top 50 pages each says to fix first, how many are actually declining?

**Method:** this cell reads the results file the pipeline just wrote and lines up the hand-written rule against the three models it trained. For each scorer it shows Precision@50, how many of the top 50 that is, how it compares with the rule, and ROC-AUC (how well the scorer orders declining pages above the others across the whole list, where 0.5 is a coin flip). It also shows how the data was split, because a model is only trusted on pages it never saw.


In [ ]:
with open("outputs/model_results.json") as handle:
    results = json.load(handle)
rule = results["baseline"]
rows = [("Hand-written rule", rule["baseline_precision_at_50"], rule["baseline_roc_auc"])]
rows += [(name.replace("_", " ").capitalize(), m["precision_at_50"], m["roc_auc"]) for name, m in results["models"].items()]
board = pd.DataFrame(rows, columns=["Scorer", "p50", "auc"]).set_index("Scorer")

table = pd.DataFrame({
    "Precision@50": board["p50"].map("{:.2f}".format),
    "Right in the top 50": (board["p50"] * 50).round().astype(int).astype(str) + " of 50",
    "Versus the rule": (board["p50"] / board.loc["Hand-written rule", "p50"]).map("{:.1f}x".format),
    "ROC-AUC": board["auc"].map("{:.3f}".format),
})
show_table("Hand-written rule versus the three trained models", table)
show("How the pages were split", {
    "validation split": results["split_strategy"].replace("_", " "),
    "pages used to train": f"{results['train_rows']:,}",
    "pages held out to test": f"{results['test_rows']:,}",
    "share of pages that are declining": f"{results['target_positive_rate']:.1%}",
    "best model (chosen on Precision@50)": results["best_model"]["name"].replace("_", " "),
})

You just ran a real ML system on real search data and saw a learned ranking beat a fixed rule. Now open `outputs/model_report.md` and skim it — that Markdown report is the _shape_ of what your own capstone should produce.


## 2. Discover a real truth yourself

The safest, most satisfying early wins are **things you find in the data** — un-leakable, and they _are_ the core lesson. Run the three cells below. Each is ~10 lines of pandas and each overturns a common SEO belief.

Every number is **computed live from the shipped CSV** — nothing is hardcoded.


In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")
df.head(3)

### Discovery A — "High search volume means more traffic." Does it?


In [ ]:
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"Correlation between search_volume and impressions_90d: {corr:.3f}")
print("Near zero -> keyword search volume barely predicts the traffic a page actually gets.")

### Discovery B — the CTR cliff by position

Click-through rate is not flat: it collapses as you move down the results.


In [ ]:
visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print(ctr_by_pos.round(4).to_string())
ctr_by_pos.plot(kind="bar", title="Mean CTR by position tier (impressions >= 100)", ylabel="CTR");

### Discovery C — is longer content the lever?

Compare word count for **declining** vs **growing** pages.


In [ ]:
wc = df.groupby("trend_direction")["word_count"].median()
print(wc.round(0).to_string())
print("\n'down' vs 'up' pages have almost the same median word count -> length is not the lever.")

## 3. 🔧 Your turn

Pick **one** of these and write a few lines below. This is your Week-1 discovery — you'll reference it in your Week-1 research-question write-up (on the InternHQ board).

- Redo Discovery A but only for pages with `impressions_90d > 0` — does the correlation change?
- In Discovery B, which `content_type` has the worst CTR _at the same position tier_?
- Find another belief to test: does `content_age_days` relate to `trend_direction`? Does `avg_position` relate to `ctr`?

**Rules:** describe what you observe as _observed / directional_ — never "I proved Google's algorithm." Keep client data out of anything you publish.


In [ ]:
# Your discovery here


### Save your work

**Colab:** _File → Save a copy in GitHub_ (writes to your own repo — that's your submission) and _File → Save a copy in Drive_ (so the session doesn't evaporate).

Next: `02_your_first_readable_model.ipynb` — where the model becomes a rule you can read.
